# 準備演習 01: エスカレーションロジックを備えたマルチツールエージェント

## 目的

- 3〜4 個のツールをどう切り分けると選択精度が上がるかを体験する
- `stop_reason` を見て `tool_use` と `end_turn` を分岐するエージェントループを段階的に作る
- 構造化エラー、リトライ、ビジネスルールガード、エスカレーションを 1 つの流れで確認する
- 複数の懸念を含むリクエストを分解し、統一レスポンスへ再統合する

## 対象ドメイン

- Domain 1: Agentic Architecture & Orchestration
- Domain 2: Tool Design & MCP Integration
- Domain 5: Context Management & Reliability

## 完成イメージ

この Notebook では **学習用の最小ループ** を純粋な Python で積み上げます。実務の実装は Claude Agent SDK を前提とし、
最新の API・hooks・MCP server の書き方は次を参照してください。

- 公式: `https://platform.claude.com/docs/en/agent-sdk/overview`
- 公式: `https://platform.claude.com/docs/en/agent-sdk/hooks`
- 完成版 Lab: [../labs/01-support-agent/](../labs/01-support-agent/)

> 重要: Notebook の最終コードは Lab と完全一致しなくて構いません。Notebook は理解の足場、Lab は reference implementation です。

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pprint import pprint

REFUND_THRESHOLD = 10000


def section(title: str):
    print(f"\n=== {title} ===")

## Step 1. ツール境界を定義する

まずは 4 つのツールを定義します。`lookup_order` と `lookup_order_summary` は機能が近く、
**description をどう書くかで選択精度が変わる** ことを観察しやすい組み合わせです。

In [ ]:
TOOLS = [
    {
        "name": "get_customer",
        "description": "顧客IDから本人確認済みの顧客プロフィールを取得する。返金・注文照会の前提確認に使う。",
        "inputs": ["customer_id"],
        "boundary": "本人確認を伴う顧客情報取得のみ。注文検索や返金処理は行わない。",
    },
    {
        "name": "lookup_order",
        "description": "注文IDで完全な注文詳細を取得する。商品明細、配送状態、返金可否を確認したい時に使う。",
        "inputs": ["order_id"],
        "boundary": "詳細確認専用。短い要約や返金実行は行わない。",
    },
    {
        "name": "lookup_order_summary",
        "description": "注文IDで要約のみを返す。ステータスと合計金額だけを素早く確認したい時に使う。",
        "inputs": ["order_id"],
        "boundary": "要約専用。商品明細や返金可否までは返さない。",
    },
    {
        "name": "process_refund",
        "description": "返金可能な注文に対して返金を実行する。副作用があり、高額返金は人間承認へエスカレーションする。",
        "inputs": ["order_id", "amount", "reason"],
        "boundary": "返金実行のみ。本人確認や注文詳細取得は事前に別ツールで行う。",
    },
]

section("定義したツール")
pprint(TOOLS)

### 確認ポイント

- `lookup_order` と `lookup_order_summary` は似ているが、**詳細確認** と **軽量要約** を description で分離できているか
- `process_refund` のような副作用ツールは、前提条件とガード条件を説明文でも明示できているか

In [ ]:
KEYWORDS = {
    "detail": {"詳細", "明細", "返金可否", "配送状態"},
    "summary": {"要約", "概要", "ざっくり", "合計だけ"},
}


def choose_lookup_tool(user_request: str) -> str:
    detail_score = sum(word in user_request for word in KEYWORDS["detail"])
    summary_score = sum(word in user_request for word in KEYWORDS["summary"])
    return "lookup_order" if detail_score >= summary_score else "lookup_order_summary"

section("description の差が選択に効くか")
for request in [
    "ORD-002 の詳細と返金可否を見たい",
    "ORD-002 の概要だけ教えて",
]:
    print(request, "->", choose_lookup_tool(request))

## Step 2. `stop_reason` で進行を制御する

ここでは Agent SDK の内部ループを **学習用に見える化した最小版** を作ります。
Notebook では `tool_use` と `end_turn` を明示的に扱い、どの分岐が必要かを確認します。

In [ ]:
CUSTOMERS = {
    "CUST-001": {"name": "田中 太郎", "is_verified": True},
}
ORDERS = {
    "ORD-002": {"status": "delivered", "total": 6000, "can_refund": True},
}

state = {"customer_verified": False, "verified_customer_id": None}


def get_customer(customer_id: str):
    customer = CUSTOMERS.get(customer_id)
    if not customer:
        return {"isError": True, "errorCategory": "validation", "isRetryable": False, "message": f"Customer not found: {customer_id}"}
    state["customer_verified"] = True
    state["verified_customer_id"] = customer_id
    return {"isError": False, "customer": customer}


def lookup_order(order_id: str):
    order = ORDERS.get(order_id)
    if not order:
        return {"isError": True, "errorCategory": "validation", "isRetryable": False, "message": f"Order not found: {order_id}"}
    return {"isError": False, "order": order}


def fake_model(messages: list[dict], user_request: str):
    if not state["customer_verified"]:
        return {"stop_reason": "tool_use", "tool": "get_customer", "input": {"customer_id": "CUST-001"}}
    if not any(msg.get("tool") == "lookup_order" for msg in messages if msg["role"] == "tool"):
        return {"stop_reason": "tool_use", "tool": "lookup_order", "input": {"order_id": "ORD-002"}}
    return {"stop_reason": "end_turn", "response": "顧客確認と注文確認が完了しました。返金可否を説明できます。"}


def run_loop(user_request: str):
    messages = [{"role": "user", "content": user_request}]
    for turn in range(1, 6):
        decision = fake_model(messages, user_request)
        print(f"turn={turn} stop_reason={decision['stop_reason']}")
        if decision["stop_reason"] == "tool_use":
            tool_name = decision["tool"]
            tool_input = decision["input"]
            payload = globals()[tool_name](**tool_input)
            messages.append({"role": "tool", "tool": tool_name, "content": payload})
            continue
        if decision["stop_reason"] == "end_turn":
            return decision["response"], messages
        raise ValueError(f"unexpected stop_reason: {decision['stop_reason']}")
    raise RuntimeError("max turns exceeded")

section("tool_use と end_turn の分岐")
final_text, loop_messages = run_loop("注文を見て返金できるか判断して")
print(final_text)

### 確認ポイント

- `tool_use` の時だけツールを実行し、結果を会話に戻しているか
- `end_turn` の時に最終レスポンスを返してループを止めているか
- 実際の Agent SDK ではこのループの大部分を SDK が管理し、Notebook では分岐の意味を理解する

## Step 3. 構造化エラーを追加する

次に、ツール失敗を単なる文字列ではなく、`errorCategory`・`isRetryable`・人間向け説明を持つ構造化エラーとして扱います。

In [ ]:
@dataclass
class ToolErrorPolicy:
    action: str
    user_message: str


POLICY = {
    "transient": ToolErrorPolicy("retry", "一時的なエラーなので自動再試行します。"),
    "validation": ToolErrorPolicy("explain", "入力値が不正なため修正をお願いします。"),
    "permission": ToolErrorPolicy("explain", "権限不足のため実行できません。"),
    "business_rule": ToolErrorPolicy("escalate", "業務ルールにより人間承認へ切り替えます。"),
}

TRANSIENT_FAILURE_ONCE = {"lookup_order": True}


def lookup_order(order_id: str):
    if TRANSIENT_FAILURE_ONCE["lookup_order"]:
        TRANSIENT_FAILURE_ONCE["lookup_order"] = False
        return {
            "isError": True,
            "errorCategory": "transient",
            "isRetryable": True,
            "message": "Inventory backend timed out",
            "humanExplanation": "在庫系のバックエンド応答が一時的に遅延しました。",
        }
    order = ORDERS.get(order_id)
    return {"isError": False, "order": order}


def handle_tool_result(result: dict):
    if not result.get("isError"):
        return "continue", "ツール成功"
    policy = POLICY[result["errorCategory"]]
    return policy.action, policy.user_message + " / " + result["humanExplanation"]

section("エラーカテゴリごとの分岐")
first_try = lookup_order("ORD-002")
print(handle_tool_result(first_try))
second_try = lookup_order("ORD-002")
print(handle_tool_result(second_try))

### 確認ポイント

- 一時的エラーは `retry`
- バリデーション・権限不足は `explain`
- 業務ルール違反は `escalate`

このように **エラーの意味を JSON で返す** と、後段の制御が安定します。

## Step 4. ビジネスルールをコードで強制し、エスカレーションへ迂回する

高額返金のような条件は prompt だけで縛らず、**プログラム的ガード** として実装します。

In [ ]:
def escalate_to_human(reason: str, context: dict):
    return {
        "isError": False,
        "handoff": {
            "reason": reason,
            "context": context,
            "queue": "refund-approval",
        },
    }


def process_refund(order_id: str, amount: float, reason: str):
    if not state["customer_verified"]:
        return {
            "isError": True,
            "errorCategory": "validation",
            "isRetryable": True,
            "message": "Customer verification is required before refund",
            "humanExplanation": "先に get_customer で本人確認してください。",
        }
    if amount > REFUND_THRESHOLD:
        return {
            "isError": True,
            "errorCategory": "business_rule",
            "isRetryable": False,
            "message": f"Refund {amount} exceeds threshold {REFUND_THRESHOLD}",
            "humanExplanation": "高額返金は自動処理できないため承認フローへ回します。",
        }
    return {"isError": False, "status": "approved", "order_id": order_id, "amount": amount}


def guarded_refund(order_id: str, amount: float, reason: str):
    result = process_refund(order_id, amount, reason)
    action, explanation = handle_tool_result(result)
    if action == "escalate":
        return escalate_to_human(explanation, {"order_id": order_id, "amount": amount})
    return result

section("閾値超え返金はエスカレーション")
state["customer_verified"] = True
pprint(guarded_refund("ORD-002", 12000, "damaged item"))

## Step 5. 複数の懸念を分解して再統合する

最後に、1 つのユーザーメッセージに「注文確認」と「返金希望」が混在しているケースを扱います。

In [ ]:
def decompose_request(user_request: str) -> list[str]:
    concerns = []
    if "注文" in user_request:
        concerns.append("order_status")
    if "返金" in user_request:
        concerns.append("refund")
    return concerns


def resolve_concern(concern: str):
    if concern == "order_status":
        order = ORDERS["ORD-002"]
        return f"注文 ORD-002 は {order['status']}、合計 {order['total']} 円です。"
    if concern == "refund":
        decision = guarded_refund("ORD-002", 12000, "customer request")
        if "handoff" in decision:
            return "返金は高額のため refund-approval キューへエスカレーションしました。"
        return "返金を自動承認しました。"
    return "未対応の concern です。"


def unify_response(user_request: str):
    concerns = decompose_request(user_request)
    parts = [resolve_concern(concern) for concern in concerns]
    return {"concerns": concerns, "response": " ".join(parts)}

section("複数懸念の統合")
pprint(unify_response("注文状況を確認して、可能なら返金も進めてください"))

## 完成版 Lab 参照

- Notebook では **1 ファイルで段階的に挙動を理解する** ことを優先しました
- 完成版 Lab では `@tool`、`create_sdk_mcp_server()`、`HookMatcher`、`PreToolUse` / `PostToolUse` を使った構成に整理されています
- 詳細は [../labs/01-support-agent/](../labs/01-support-agent/) を参照してください
- API や hook の最新形は、必ず Claude Agent SDK 公式ドキュメントで確認してください